git config --global init.defaultBranch main


git config --global --get init.defaultBranch


## 1. Check if you already have SSH keys

In WSL Ubuntu:

```bash
ls -al ~/.ssh
```

Look for files like:

- `id_ed25519`
- `id_ed25519.pub`

If you already have them and want to reuse them, skip to Step 4.


## 2. Generate a new SSH key

Recommended modern key type:

```bash
ssh-keygen -t ed25519 -C "your_email@example.com"
```

When prompted:

```text
Enter file in which to save the key:
```

Press Enter to use default:

```text
~/.ssh/id_ed25519
```

Then optionally add a passphrase.


## 3. Start the SSH agent and add the key

Run:

```bash
eval "$(ssh-agent -s)"
```

Then:

```bash
ssh-add ~/.ssh/id_ed25519
```

If you get “Could not open a connection to your authentication agent”, run the first command again.


## 4. Copy the public key

Show the key:

```bash
cat ~/.ssh/id_ed25519.pub
```

Copy the entire output beginning with:

```text
ssh-ed25519 ...
```


## 5. Add the key to GitHub

Open:

[GitHub SSH Settings](https://github.com/settings/keys)

Then:

1. Click **New SSH key**
2. Give it a title like:
   - `WSL Ubuntu Laptop`

3. Paste the public key
4. Save


## 6. Test the connection

Run:

```bash
ssh -T git@github.com
```

First time you’ll see:

```text
Are you sure you want to continue connecting (yes/no/[fingerprint])?
```

Type:

```text
yes
```

If successful:

```text
Hi <username>! You've successfully authenticated...
```


# Git Mastery for DevOps

This notebook transforms you from a complete Git beginner into a confident Git practitioner on a professional DevOps team. You will learn exclusively through one continuous project: **devops-notes-portal**, a documentation repository for Linux, Docker, Nginx, Kubernetes, CI/CD, and infrastructure notes.

Every command you type evolves this repository. No disconnected examples. No theory dumps. Just real, terminal-driven, production‑oriented labs.

**Your prerequisites** are met: you have WSL Ubuntu, Git installed, a GitHub account, an SSH key configured, and basic Linux command knowledge. We start with Git immediately.


# Module 1 — Git Fundamentals

## Why This Exists

Before you type a single Git command, you need a mental model. Without it, Git feels like magic—and magic breaks when you need it most. This module builds that model.


## Lab: The Problem Without Git

1. Create a directory to simulate manual version control:

```bash
mkdir manual-vcs && cd manual-vcs
echo "server = nginx01" > config.txt
cp config.txt config_v1.txt
echo "server = nginx02" >> config.txt
cp config.txt config_v2.txt
ls
```

**Expected output:**

```text
config.txt  config_v1.txt  config_v2.txt
```

This manual approach fails quickly: no history, no collaboration, no idea who changed what.


## What Git Solves

Git records snapshots of your entire project over time. It tracks **who**, **what**, **when**, and **why**.

- **Version control** – every committed state is recoverable.
- **Collaboration** – multiple people can work without overwriting each other.
- **Traceability** – every change has an author, timestamp, and message.
- **Branching / merging** – experiment safely, integrate cleanly.


## The Four Areas of Git

Everything you do in Git involves four zones:

```text
  Working          Staging Area        Local Repository     Remote
  Directory         (Index)              (.git)            Repository
 ┌─────────┐      ┌─────────┐         ┌─────────┐        ┌─────────┐
 │         │      │         │         │         │        │  GitHub │
 │  files  │─────►│  files  │────────►│ commits │───────►│  etc.   │
 │         │ add  │         │ commit  │         │ push   │         │
 └─────────┘      └─────────┘         └─────────┘        └─────────┘
     ▲                                   │
     │        restore / checkout         │
     └───────────────────────────────────┘
```

- **Working directory** – the files you see and edit.
- **Staging area (index)** – a preparation zone where you build the next commit.
- **Local repository (.git)** – the permanent history of commits.
- **Remote repository** – a shared copy (e.g., on GitHub).


## Lab: Visualising the Four Areas

No commands yet. Just picture the flow:

1. Edit a file → it’s in the _working directory_.
2. `git add` → move it to the _staging area_.
3. `git commit` → save it as a permanent snapshot in the _local repository_.
4. `git push` → publish to the _remote repository_.

We’ll execute this in the next module.


## Verify

Explain in your own words what happens when you run `git add` and `git commit`. Write it down before moving on.


## Challenge

Draw (on paper) a box diagram of the four areas and draw arrows for `add`, `commit`, `push`, and `pull`. Label each zone.


## Common Mistakes

- Believing the working directory is the repository. It is only a view.
- Thinking `git add` saves a version. Only `git commit` does that.
- Confusing the staging area with a temporary backup. It’s a deliberate _commit blueprint_.


## Mini Quiz

1. Which command moves changes from the working directory to the staging area?  
   **Answer:** `git add`

2. Which area stores committed snapshots permanently?  
   **Answer:** Local repository (`.git` directory)

3. True or False: `git push` sends changes from the staging area to GitHub.  
   **Answer:** False – it sends _commits_ from the local repository.


## Real DevOps Usage

In CI/CD pipelines, the four areas map directly to automation steps:

- Working directory → where the pipeline checks out code.
- Staging area → not used directly by pipelines, but what `git add` simulates before a script commits.
- Local repository → where the pipeline can create tags, cherry‑pick, or revert.
- Remote → the source of truth that triggers builds.

Understanding these zones prevents mistakes like “the pipeline built the wrong code because I forgot to push”.


# Module 2 — Project Setup

## Why This Exists

Every project begins with `git init` and `git config`. Misconfigured author information leads to unreadable history. Doing it right from the start is a professional habit.


## Lab: Create the Repository

1. Create and enter the project directory:

```bash
mkdir devops-notes-portal
cd devops-notes-portal
git init
```

**Expected output:**

```text
Initialized empty Git repository in /home/user/devops-notes-portal/.git/
```

2. Check the status:

```bash
git status
```

```text
On branch main

No commits yet

nothing to commit (create/copy files and use "git add" to track)
```

3. Set local repository configuration:

```bash
git config user.name "DevOps Student"
git config user.email "student@devops-lab.local"
```

4. Verify:

```bash
git config --list --local
```

```text
user.name=DevOps Student
user.email=student@devops-lab.local
core.repositoryformatversion=0
core.filemode=true
core.bare=false
...
```

5. (Optional) Set global defaults if not already done:

```bash
git config --global user.name "Your Real Name"
git config --global user.email "your-real@email.com"
```


## Verify

Run `cat .git/config` to see the local configuration section `[user]`.


## Challenge

Create a second repository in `/tmp/test-repo`, initialize it, set a different `user.name` (e.g., “Ops Bot”), and confirm it’s stored locally. This shows that `git config` without `--global` affects only that repository.


## Common Mistakes

- Forgetting to set `user.email`. Commits without an email can cause hooks and CI checks to fail.
- Using `--global` for a project that should have a specific identity (e.g., a bot account).


## Troubleshooting

**“fatal: not a git repository”**  
You are not inside a directory that contains `.git`. Run `git init` first.

**“\*** Please tell me who you are.”\*\*  
Set `user.name` and `user.email` before committing.


## Mini Quiz

1. Which flag shows only the local repository configuration?  
   **Answer:** `--local`

2. Where is the local configuration stored?  
   **Answer:** `.git/config`


# Module 3 — First Commits

## Why This Exists

Commits are the core of Git. Every DevOps action—code reviews, rollbacks, audits—relies on well‑crafted commits. You will now create your first professional commits in the DevOps Notes Portal.


## Lab: Your First Commit

1. Create `README.md`:

```bash
echo "# DevOps Notes Portal" > README.md
echo "Central documentation for Linux, Docker, Nginx, Kubernetes, and CI/CD." >> README.md
```

2. Stage and commit:

```bash
git add README.md
git commit -m "feat: initial project structure with README"
```

**Expected output:**

```text
[main (root-commit) a3f1c2e] feat: initial project structure with README
 1 file changed, 2 insertions(+)
 create mode 100644 README.md
```

3. Add Linux notes:

```bash
mkdir notes
echo "# Linux Notes" > notes/linux.md
echo "## Common Commands" >> notes/linux.md
echo "- ls - list files" >> notes/linux.md
echo "- df - disk free" >> notes/linux.md
```

```bash
git add notes/linux.md
git commit -m "feat: add Linux notes section"
```

4. Add Docker notes:

```bash
echo "# Docker Notes" > notes/docker.md
echo "## Basic Commands" >> notes/docker.md
echo "- docker run" >> notes/docker.md
echo "- docker ps" >> notes/docker.md
```

```bash
git add notes/docker.md
git commit -m "feat: add Docker notes section"
```


## Verify History

```bash
git log
```

```text
commit b7e3f4d (HEAD -> main)
Author: DevOps Student <student@devops-lab.local>
Date:   ...

    feat: add Docker notes section

commit d1a8c9b
Author: DevOps Student <student@devops-lab.local>
Date:   ...

    feat: add Linux notes section

commit a3f1c2e
Author: DevOps Student <student@devops-lab.local>
Date:   ...

    feat: initial project structure with README
```


## Show a Specific Commit

```bash
git show b7e3f4d
```

(Use the actual hash from your log.) Shows full diff and metadata.


## Challenge

1. Create `notes/nginx.md` with a title and two bullet points.
2. Stage and commit it with message `docs: add Nginx notes placeholder`.
3. Use `git log --oneline` to see a compact history.

**Expected output (after challenge):**

```text
e5f6a7g (HEAD -> main) docs: add Nginx notes placeholder
b7e3f4d feat: add Docker notes section
d1a8c9b feat: add Linux notes section
a3f1c2e feat: initial project structure with README
```


## Common Mistakes

- Skipping `git add` and wondering why the file isn’t committed.
- Using `git commit -a` (stage all) too early. Explicit staging builds discipline.
- Writing vague messages like “update”. DevOps colleagues will thank you for clarity.


## Mini Quiz

1. What does `git show` without arguments display?  
   **Answer:** The latest commit (HEAD) details.

2. Which command gives a one‑line‑per‑commit log?  
   **Answer:** `git log --oneline`


# Module 4 — Git Internals (Hands‑on)

## Why This Exists

Understanding Git’s object model demystifies everything else. When you see a hash, you know exactly what it represents. This knowledge is crucial for debugging, recovering lost data, and writing advanced hooks.


## The Object Model

Git stores everything as objects in `.git/objects`. Three types:

- **Blob** – file content (not the file name).
- **Tree** – directory structure (names and references to blobs / trees).
- **Commit** – snapshot metadata: author, message, parent commit(s), and a root tree.

A commit points to a tree, which points to blobs.

```
 Commit (a3f1c2e)
   │
   tree (4b8e5a)
   ├── blob (f7a3)  "README.md" content
   ├── tree (c2d1)  "notes"
   │    ├── blob (b3e9) "linux.md"
   │    └── blob (d4f1) "docker.md"
```


## Lab: Peek Inside Objects

1. List objects in the database:

```bash
find .git/objects -type f
```

**Example output:**

```text
.git/objects/a3/f1c2e...
.git/objects/b7/e3f4d...
...
```

2. Use `git cat-file` to inspect the latest commit:

```bash
git cat-file -p HEAD
```

**Expected output:**

```text
tree 4b8e5a...
parent d1a8c9b...   (if not root)
author DevOps Student <student@devops-lab.local> ...
committer DevOps Student <student@devops-lab.local> ...

docs: add Nginx notes placeholder
```

3. Follow the tree:

```bash
git cat-file -p 4b8e5a   # replace with your tree hash
```

```text
100644 blob f7a3...    README.md
040000 tree c2d1...    notes
```

4. Follow a blob:

```bash
git cat-file -p f7a3   # replace with blob hash
```

```text
# DevOps Notes Portal
Central documentation...
```

5. Compute hash of a file manually (same as Git’s internal hash):

```bash
git hash-object README.md
```

```text
f7a3...
```

Compare with the blob hash from the tree. They match.


## Verify

Run `git ls-files --stage` to see the current staging area content. This shows the blob hashes for each tracked file.


## Challenge

1. Find the commit hash of the first commit (the root).
2. Use `git cat-file -p` on that commit’s tree.
3. List all blobs inside that tree. Confirm that no `nginx.md` exists yet.
4. Explain why the root commit has no `parent` line.


## Common Mistakes

- Confusing blob hash with file name. Git tracks _content_, not names.
- Editing a file and expecting the hash to change before `git add`. The working‑directory copy is not yet hashed.


## Mini Quiz

1. What type of Git object represents a directory?  
   **Answer:** Tree

2. If you change only the file’s name, does the blob hash change?  
   **Answer:** No—the content (blob) stays the same; the tree entry changes.


# Module 5 — Tracking Changes

## Why This Exists

Real work happens in the working directory. You need to see what you’ve modified, compare versions, and selectively discard or stage changes. `git diff` and `git restore` are your daily tools.


## Lab: Explore Diffs and Restoration

1. Modify `notes/linux.md`:

```bash
echo "## Disk Management" >> notes/linux.md
echo "- du - disk usage" >> notes/linux.md
```

2. View unstaged changes:

```bash
git diff
```

**Partial output:**

```diff
diff --git a/notes/linux.md b/notes/linux.md
index b3e9..c5f2 100644
--- a/notes/linux.md
+++ b/notes/linux.md
@@ -2,2 +2,4 @@
...
+## Disk Management
+- du - disk usage
```

3. Stage the change:

```bash
git add notes/linux.md
```

4. View staged changes:

```bash
git diff --staged
```

Same diff, but now it’s ready to be committed.

5. Commit it:

```bash
git commit -m "docs: add disk management section to Linux notes"
```

6. Discard a working‑directory change (restore a file to its last committed state):

```bash
echo "Temporary debug line" >> notes/docker.md
git diff
git restore notes/docker.md
```

Now `notes/docker.md` is clean again.


## Old‑style `git checkout`

If you encounter `git checkout -- <file>`, it is the older equivalent of `git restore <file>`. Prefer `git restore` for clarity.


## Challenge

1. Create a new file `notes/kubernetes.md` with some content. Do **not** stage it.
2. Run `git diff` and examine the output (new file shows as `/dev/null`).
3. Stage it and observe the diff again with `--staged`.
4. **Unstage** it using `git restore --staged notes/kubernetes.md` (then delete it or keep for later).


## Common Mistakes

- Running `git diff` after `git add` and seeing nothing. Use `--staged` to see staged changes.
- Using `git checkout` to restore a file, which can be confused with branch switching. Use `git restore`.


## Troubleshooting

**“error: pathspec 'file' did not match any file(s) known to git”**  
The file does not exist in the repository yet. Add it first or check the path.


## Mini Quiz

1. How do you see the difference between the staging area and the last commit?  
   **Answer:** `git diff --staged`

2. Which command discards all local changes in a specific file, reverting to the last commit?  
   **Answer:** `git restore <file>`
